In [2]:
import sys
from dotenv import load_dotenv

load_dotenv(r"c:\Users\admin\demand-forecasting\.env")

sys.path.append(r"c:\Users\admin\demand-forecasting")

In [3]:
from etl.load import get_engine
import pandas as pd

engine = get_engine()

In [4]:
query_last_sale_date = '''
    SELECT MAX(d.la_date)
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
'''
last_sale_date = pd.read_sql(query_last_sale_date, engine).iloc[0, 0]
print(last_sale_date)

cutoff_date = last_sale_date - pd.Timedelta(days=50)
print(cutoff_date)

2016-04-24
2016-03-05


In [5]:
query_train = f'''
    SELECT f.item_id, f.store_id, f.quantite, d.la_date
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
    WHERE d.la_date < '{cutoff_date}'
'''

query_test = f'''
    SELECT f.item_id, f.store_id, f.quantite, d.la_date
    FROM fact_sales f
    JOIN dim_date d ON f.date_id = d.id
    WHERE d.la_date >= '{cutoff_date}'
'''

train = pd.read_sql(query_train, engine)
test = pd.read_sql(query_test, engine)

print(train.shape, test.shape)

(56772380, 4) (1554990, 4)


In [6]:
# Dernier jour de chaque série (item+store) dans train, par jour de semaine
train['la_date'] = pd.to_datetime(train['la_date'])
train['jour_semaine'] = train['la_date'].dt.dayofweek  # 0=lundi, 6=dimanche

# Pour chaque item+store+jour_de_semaine, la dernière valeur connue dans train
last_known = (
    train
    .sort_values('la_date')
    .groupby(['item_id', 'store_id', 'jour_semaine'])
    .last()
    .reset_index()[['item_id', 'store_id', 'jour_semaine', 'quantite']]
    .rename(columns={'quantite': 'prediction_naive_saisonnier'})
)

test['la_date'] = pd.to_datetime(test['la_date'])
test['jour_semaine'] = test['la_date'].dt.dayofweek

test_with_pred = test.merge(
    last_known,
    on=['item_id', 'store_id', 'jour_semaine'],
    how='left'
)

print(test_with_pred.shape)
print(test_with_pred.isna().sum())
print(test_with_pred.head())

(1554990, 6)
item_id                        0
store_id                       0
quantite                       0
la_date                        0
jour_semaine                   0
prediction_naive_saisonnier    0
dtype: int64
   item_id  store_id  quantite    la_date  jour_semaine  \
0        1         1         0 2016-03-05             5   
1        2         1         0 2016-03-05             5   
2        3         1         6 2016-03-05             5   
3        4         1         1 2016-03-05             5   
4        5         1         1 2016-03-05             5   

   prediction_naive_saisonnier  
0                            4  
1                            0  
2                            0  
3                            0  
4                            1  


In [7]:
# Dernière valeur connue (dernier jour) pour chaque item+store dans train
last_value = (
    train
    .sort_values('la_date')
    .groupby(['item_id', 'store_id'])
    .last()
    .reset_index()[['item_id', 'store_id', 'quantite']]
    .rename(columns={'quantite': 'prediction_naive_simple'})
)

test_with_pred = test_with_pred.merge(
    last_value,
    on=['item_id', 'store_id'],
    how='left'
)

print(test_with_pred[['quantite', 'prediction_naive_saisonnier', 'prediction_naive_simple']].head())
print(test_with_pred.isna().sum())

   quantite  prediction_naive_saisonnier  prediction_naive_simple
0         0                            4                        1
1         0                            0                        0
2         6                            0                        0
3         1                            0                        0
4         1                            1                        0
item_id                        0
store_id                       0
quantite                       0
la_date                        0
jour_semaine                   0
prediction_naive_saisonnier    0
prediction_naive_simple        0
dtype: int64


In [8]:
def wape(y_true, y_pred):
    return (y_true - y_pred).abs().sum() / y_true.sum()

wape_naive_simple = wape(test_with_pred['quantite'], test_with_pred['prediction_naive_simple'])
wape_naive_saisonnier = wape(test_with_pred['quantite'], test_with_pred['prediction_naive_saisonnier'])

print(f"WAPE naïf simple : {wape_naive_simple:.4f}")
print(f"WAPE naïf saisonnier : {wape_naive_saisonnier:.4f}")

WAPE naïf simple : 0.9042
WAPE naïf saisonnier : 0.9160
